BirdCLEF 2026 — Wildlife Sound Classification
Workflow:

Section 1 — Run once to train and save models (cnn_best.pth, hybrid_best.pth, label_encoder.pkl).
Section 2 — Run every time you reopen the notebook to load saved models and evaluate — no retraining needed.
Set TRAIN_MODE = True (Section 1 cell) only when you want to retrain.

In [1]:
"""
BirdCLEF 2026 - Wildlife Sound Classification Pipeline
======================================================
A comprehensive solution for multi-species audio classification
"""

import os
import random
import pickle
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union
from dataclasses import dataclass
from collections import defaultdict

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import autocast, GradScaler
import torchaudio
import torchaudio.transforms as T

import timm
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
import albumentations as A

warnings.filterwarnings("ignore")

# ============================================================================
# CONFIGURATION
# ============================================================================

@dataclass
class Config:
    """Central configuration for the pipeline"""
    # Paths
    data_dir: Path = Path("/kaggle/input/competitions/birdclef-2026")
    output_dir: Path = Path("/kaggle/working")

    # Audio parameters
    sample_rate: int = 32000
    segment_duration: float = 5.0
    hop_length: int = 320
    n_fft: int = 2048
    n_mels: int = 128
    fmin: int = 20
    fmax: int = 16000

    # Training parameters
    batch_size: int = 64
    num_workers: int = 2
    epochs: int = 3
    learning_rate: float = 2e-4
    weight_decay: float = 1e-5

    # Model parameters
    model_name: str = "tf_efficientnet_b0_ns"
    num_classes: int = 234

    # Augmentation
    mixup_alpha: float = 0.5
    use_mixup: bool = True

    # Misc
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    @property
    def segment_samples(self) -> int:
        return int(self.segment_duration * self.sample_rate)


config = Config()


def set_seed(seed: int = 42):
    """Set random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(config.seed)
print(f"Device: {config.device}")
print(f"Data dir: {config.data_dir}")
print(f"Output dir: {config.output_dir}")


Device: cuda
Data dir: /kaggle/input/competitions/birdclef-2026
Output dir: /kaggle/working


/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()


In [2]:
# ============================================================================
# PART 1: DATA CLEANING
# ============================================================================

class DataCleaner:
    """Clean and validate the competition data"""

    def __init__(self, config: Config):
        self.config = config

    def load_metadata(self) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        print("Loading metadata...")
        train_df = pd.read_csv(self.config.data_dir / "train.csv")
        print(f"  Train CSV: {len(train_df)} recordings")
        taxonomy_df = pd.read_csv(self.config.data_dir / "taxonomy.csv")
        print(f"  Taxonomy: {len(taxonomy_df)} species")
        soundscape_labels_path = self.config.data_dir / "train_soundscapes_labels.csv"
        if soundscape_labels_path.exists():
            soundscape_labels_df = pd.read_csv(soundscape_labels_path)
            print(f"  Soundscape labels: {len(soundscape_labels_df)} segments")
        else:
            soundscape_labels_df = pd.DataFrame()
            print("  Soundscape labels: Not found")
        return train_df, taxonomy_df, soundscape_labels_df

    def clean_train_data(self, df: pd.DataFrame) -> pd.DataFrame:
        print("\nCleaning training data...")
        initial_count = len(df)
        df = df.drop_duplicates(subset=["filename"])
        df["rating"] = df["rating"].fillna(0)
        df["secondary_labels"] = df["secondary_labels"].fillna("[]")
        df["latitude"] = df["latitude"].fillna(0)
        df["longitude"] = df["longitude"].fillna(0)
        df["secondary_labels_list"] = df["secondary_labels"].apply(
            lambda x: eval(x) if isinstance(x, str) and x.startswith("[") else []
        )
        df = df[(df["rating"] >= 3) | (df["rating"] == 0)]
        print(f"  Cleaned: {initial_count} -> {len(df)} recordings")
        return df

    def analyze_class_distribution(self, df: pd.DataFrame) -> pd.Series:
        class_counts = df["primary_label"].value_counts()
        print(f"\nClass distribution:")
        print(f"  Min: {class_counts.min()}  Max: {class_counts.max()}  Mean: {class_counts.mean():.1f}")
        return class_counts


# ============================================================================
# PART 2: AUDIO PREPROCESSING
# ============================================================================

class AudioProcessor:
    def __init__(self, config: Config):
        self.config = config
        self.mel_transform = T.MelSpectrogram(
            sample_rate=config.sample_rate,
            n_fft=config.n_fft,
            hop_length=config.hop_length,
            n_mels=config.n_mels,
            f_min=config.fmin,
            f_max=config.fmax,
            power=2.0,
        )
        self.db_transform = T.AmplitudeToDB(stype="power", top_db=80)

    def load_audio(self, filepath, offset: float = 0.0, duration: Optional[float] = None) -> np.ndarray:
        try:
            audio, sr = librosa.load(
                filepath, sr=self.config.sample_rate,
                offset=offset, duration=duration, mono=True
            )
            return audio
        except Exception as e:
            print(f"Error loading {filepath}: {e}")
            samples = int(self.config.sample_rate * (duration or self.config.segment_duration))
            return np.zeros(samples, dtype=np.float32)

    def normalize_audio(self, audio: np.ndarray) -> np.ndarray:
        max_val = np.abs(audio).max()
        if max_val > 0:
            audio = audio / max_val
        return audio

    def pad_or_truncate(self, audio: np.ndarray, target_length: int) -> np.ndarray:
        if len(audio) > target_length:
            start = np.random.randint(0, len(audio) - target_length)
            audio = audio[start:start + target_length]
        elif len(audio) < target_length:
            audio = np.pad(audio, (0, target_length - len(audio)), mode="constant")
        return audio

    def audio_to_melspec(self, audio: np.ndarray) -> torch.Tensor:
        waveform = torch.from_numpy(audio).float().unsqueeze(0)
        mel_spec = self.mel_transform(waveform)
        mel_spec_db = self.db_transform(mel_spec)
        mel_spec_db = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min() + 1e-6)
        return mel_spec_db.squeeze(0)

    def process_file(self, filepath, offset: float = 0.0, duration: Optional[float] = None) -> torch.Tensor:
        duration = duration or self.config.segment_duration
        audio = self.load_audio(filepath, offset=offset, duration=duration)
        audio = self.normalize_audio(audio)
        audio = self.pad_or_truncate(audio, int(self.config.sample_rate * duration))
        return self.audio_to_melspec(audio)


# ============================================================================
# PART 3: AUGMENTATIONS
# ============================================================================

class AudioAugmentation:
    def __init__(self, config: Config):
        self.config = config

    def time_shift(self, audio: np.ndarray, shift_max: float = 0.2) -> np.ndarray:
        shift = int(len(audio) * np.random.uniform(-shift_max, shift_max))
        return np.roll(audio, shift)

    def add_noise(self, audio: np.ndarray, noise_factor: float = 0.005) -> np.ndarray:
        return audio + np.random.randn(len(audio)) * noise_factor

    def change_volume(self, audio: np.ndarray, volume_range: Tuple[float, float] = (0.5, 1.5)) -> np.ndarray:
        return audio * np.random.uniform(*volume_range)


class SpecAugmentation:
    def __init__(self, config: Config):
        self.freq_mask = T.FrequencyMasking(freq_mask_param=15)
        self.time_mask = T.TimeMasking(time_mask_param=35)

    def apply(self, spec: torch.Tensor, num_masks: int = 2) -> torch.Tensor:
        for _ in range(num_masks):
            spec = self.freq_mask(spec)
            spec = self.time_mask(spec)
        return spec


def mixup(x1, y1, x2, y2, alpha: float = 0.5) -> Tuple[torch.Tensor, torch.Tensor]:
    lam = np.random.beta(alpha, alpha)
    return lam * x1 + (1 - lam) * x2, lam * y1 + (1 - lam) * y2


# ============================================================================
# PART 4: DATASETS
# ============================================================================

class BirdCLEFTrainDataset(Dataset):
    def __init__(self, df, config, label_encoder, audio_dir, augment=True):
        self.df = df.reset_index(drop=True)
        self.config = config
        self.label_encoder = label_encoder
        self.audio_dir = audio_dir
        self.augment = augment
        self.audio_processor = AudioProcessor(config)
        self.audio_aug = AudioAugmentation(config)
        self.spec_aug = SpecAugmentation(config)
        self.labels = self.label_encoder.transform(self.df["primary_label"].values)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filepath = self.audio_dir / row["filename"]
        audio = self.audio_processor.load_audio(filepath)
        audio = self.audio_processor.normalize_audio(audio)
        if self.augment:
            if np.random.random() < 0.5:
                audio = self.audio_aug.time_shift(audio)
            if np.random.random() < 0.3:
                audio = self.audio_aug.add_noise(audio)
            if np.random.random() < 0.5:
                audio = self.audio_aug.change_volume(audio)
        if len(audio) > self.config.segment_samples:
            start = np.random.randint(0, len(audio) - self.config.segment_samples)
            audio = audio[start:start + self.config.segment_samples]
        else:
            audio = self.audio_processor.pad_or_truncate(audio, self.config.segment_samples)
        mel_spec = self.audio_processor.audio_to_melspec(audio)
        if self.augment and np.random.random() < 0.5:
            mel_spec = self.spec_aug.apply(mel_spec.unsqueeze(0)).squeeze(0)
        mel_spec = mel_spec.unsqueeze(0).repeat(3, 1, 1)
        label = torch.zeros(self.config.num_classes, dtype=torch.float32)
        label[self.labels[idx]] = 1.0
        sec_labels = row.get("secondary_labels_list")
        if sec_labels is not None and isinstance(sec_labels, list):
            for sl in sec_labels:
                if sl in self.label_encoder.classes_:
                    label[self.label_encoder.transform([sl])[0]] = 1.0
        return mel_spec, label


class SoundscapeDataset(Dataset):
    def __init__(self, filepaths, config):
        self.config = config
        self.audio_processor = AudioProcessor(config)
        self.segments = []
        for fp in filepaths:
            for start_sec in range(0, 60, 5):
                self.segments.append({
                    "filepath": fp,
                    "start_sec": start_sec,
                    "end_sec": start_sec + 5,
                    "row_id": f"{fp.stem}_{start_sec + 5}"
                })

    def __len__(self):
        return len(self.segments)

    def __getitem__(self, idx):
        seg = self.segments[idx]
        mel_spec = self.audio_processor.process_file(seg["filepath"], offset=seg["start_sec"], duration=5.0)
        return mel_spec.unsqueeze(0).repeat(3, 1, 1), seg["row_id"]


# ============================================================================
# PART 5: MODELS
# ============================================================================

class AttentionPooling(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(in_features, in_features // 4), nn.ReLU(),
            nn.Linear(in_features // 4, 1),
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        weights = F.softmax(self.attention(x), dim=1)
        return (x * weights).sum(dim=1)


class BirdCLEFCNN(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.backbone = timm.create_model(
            config.model_name, pretrained=False, in_chans=3, num_classes=0, global_pool=""
        )
        with torch.no_grad():
            dummy = torch.randn(1, 3, config.n_mels, 500)
            features = self.backbone(dummy)
            self.feature_dim = features.shape[1]
        self.attention_pool = AttentionPooling(self.feature_dim)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(self.feature_dim, config.num_classes),
        )

    def forward(self, x):
        features = self.backbone(x)
        features = features.mean(dim=2)
        pooled = self.attention_pool(features)
        return self.classifier(pooled)


class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads=8, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, int(dim * mlp_ratio)), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(int(dim * mlp_ratio), dim), nn.Dropout(dropout),
        )

    def forward(self, x):
        x_norm = self.norm1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x = x + attn_out
        return x + self.mlp(self.norm2(x))


class HybridModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.cnn = timm.create_model(
            "tf_efficientnet_b0_ns", pretrained=False, in_chans=1, num_classes=0, global_pool=""
        )
        with torch.no_grad():
            dummy = torch.randn(1, 1, config.n_mels, 500)
            cnn_out = self.cnn(dummy)
            self.cnn_channels = cnn_out.shape[1]
            self.cnn_h = cnn_out.shape[2]
            self.cnn_w = cnn_out.shape[3]
        embed_dim = 256
        self.proj = nn.Conv2d(self.cnn_channels, embed_dim, kernel_size=1)
        num_positions = self.cnn_h * self.cnn_w
        self.pos_embed = nn.Parameter(torch.zeros(1, num_positions, embed_dim))
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.transformer = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads=4, mlp_ratio=4.0, dropout=0.1) for _ in range(4)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(embed_dim, config.num_classes),
        )
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        x = x[:, 0:1, :, :]
        batch_size = x.shape[0]
        features = self.cnn(x)
        features = self.proj(features)
        features = features.flatten(2).transpose(1, 2)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        features = torch.cat([cls_tokens, features], dim=1)
        pos_embed = torch.cat([
            torch.zeros(1, 1, features.shape[-1], device=features.device),
            self.pos_embed
        ], dim=1)
        features = features + pos_embed
        for block in self.transformer:
            features = block(features)
        features = self.norm(features)
        return self.head(features[:, 0])


class EnsembleModel:
    def __init__(self, models, weights=None):
        self.models = models
        self.weights = weights or [1.0 / len(models)] * len(models)

    @torch.no_grad()
    def predict(self, inputs):
        all_probs = []
        for model, weight in zip(self.models, self.weights):
            model.eval()
            probs = torch.sigmoid(model(inputs)) * weight
            all_probs.append(probs)
        return torch.stack(all_probs, dim=0).sum(dim=0)


# ============================================================================
# PART 6: LOSS FUNCTIONS
# ============================================================================

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        bce = F.binary_cross_entropy_with_logits(inputs, targets, reduction="none")
        pt = torch.exp(-bce)
        return (self.alpha * (1 - pt) ** self.gamma * bce).mean()


# ============================================================================
# PART 7: TRAINER
# ============================================================================

class Trainer:
    def __init__(self, model, config, criterion=None):
        self.model = model.to(config.device)
        self.config = config
        self.criterion = criterion or FocalLoss()
        self.optimizer = torch.optim.AdamW(
            model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay
        )
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer, T_max=config.epochs, eta_min=1e-6
        )
        self.scaler = GradScaler("cuda")
        self.train_losses = []
        self.val_losses = []
        self.best_val_loss = float("inf")

    def train_epoch(self, dataloader, epoch):
        self.model.train()
        total_loss = 0.0
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1} [Train]")
        for batch_idx, (inputs, targets) in enumerate(pbar):
            inputs = inputs.to(self.config.device)
            targets = targets.to(self.config.device)
            if self.config.use_mixup and np.random.random() < 0.5:
                perm = torch.randperm(inputs.size(0))
                inputs, targets = mixup(inputs, targets, inputs[perm], targets[perm], self.config.mixup_alpha)
            self.optimizer.zero_grad()
            with autocast("cuda"):
                outputs = self.model(inputs)
                loss = self.criterion(outputs, targets)
            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.scaler.step(self.optimizer)
            self.scaler.update()
            if torch.isnan(loss):
                print(f"NaN loss at batch {batch_idx}, skipping...")
                continue
            total_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        avg = total_loss / len(dataloader)
        self.train_losses.append(avg)
        return avg

    @torch.no_grad()
    def validate(self, dataloader):
        self.model.eval()
        total_loss = 0.0
        for inputs, targets in tqdm(dataloader, desc="Validating"):
            inputs = inputs.to(self.config.device)
            targets = targets.to(self.config.device)
            with autocast("cuda"):
                outputs = self.model(inputs)
                loss = self.criterion(outputs, targets)
            total_loss += loss.item()
        avg = total_loss / len(dataloader)
        self.val_losses.append(avg)
        return avg

    def fit(self, train_loader, val_loader, save_path):
        for epoch in range(self.config.epochs):
            train_loss = self.train_epoch(train_loader, epoch)
            val_loss = self.validate(val_loader)
            print(f"Epoch {epoch+1}/{self.config.epochs}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  lr={self.scheduler.get_last_lr()[0]:.2e}")
            self.scheduler.step()
            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                torch.save({
                    "epoch": epoch,
                    "model_state_dict": self.model.state_dict(),
                    "optimizer_state_dict": self.optimizer.state_dict(),
                    "val_loss": val_loss,
                }, save_path)
                print(f"  ✓ Saved best model (val_loss={val_loss:.4f}) → {save_path}")
        return self.model


# ============================================================================
# PART 8: INFERENCE
# ============================================================================

class Predictor:
    def __init__(self, model, config, label_encoder):
        self.model = model.to(config.device)
        self.model.eval()
        self.config = config
        self.label_encoder = label_encoder

    @torch.no_grad()
    def predict_soundscapes(self, dataloader, threshold=0.5):
        all_row_ids, all_probs = [], []
        for inputs, row_ids in tqdm(dataloader, desc="Predicting"):
            inputs = inputs.to(self.config.device)
            with autocast("cuda"):
                probs = torch.sigmoid(self.model(inputs)).cpu().numpy()
            all_row_ids.extend(row_ids)
            all_probs.append(probs)
        all_probs = np.vstack(all_probs)
        submission = pd.DataFrame({"row_id": all_row_ids})
        for i, species in enumerate(self.label_encoder.classes_):
            submission[species] = all_probs[:, i]
        return submission


# ============================================================================
# PART 9: PIPELINE HELPERS
# ============================================================================

def create_folds(df, n_folds=5, stratify_col="primary_label"):
    df = df.copy().reset_index(drop=True)
    df["fold"] = -1
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=config.seed)
    for fold, (_, val_idx) in enumerate(skf.split(df, df[stratify_col])):
        df.loc[val_idx, "fold"] = fold
    return df


def build_val_loader(val_fold_df, label_encoder, config):
    """Build a validation DataLoader from a fold DataFrame."""
    val_dataset = BirdCLEFTrainDataset(
        val_fold_df, config, label_encoder,
        config.data_dir / "train_audio", augment=False
    )
    return DataLoader(
        val_dataset, batch_size=config.batch_size,
        shuffle=False, num_workers=0, pin_memory=True
    )


def evaluate_model(model, dataloader, config):
    """Evaluate a model and return mean ROC-AUC over classes that have positive samples."""
    model.eval()
    all_probs, all_targets = [], []
    with torch.no_grad():
        for inputs, targets in tqdm(dataloader, desc="Evaluating"):
            inputs = inputs.to(config.device)
            with torch.amp.autocast("cuda"):
                probs = torch.sigmoid(model(inputs)).cpu().numpy()
            all_probs.append(probs)
            all_targets.append(targets.numpy())
    all_probs = np.vstack(all_probs)
    all_targets = np.vstack(all_targets)
    auc_scores = []
    for i in range(all_targets.shape[1]):
        if all_targets[:, i].sum() > 0:
            auc_scores.append(roc_auc_score(all_targets[:, i], all_probs[:, i]))
    mean_auc = float(np.mean(auc_scores)) if auc_scores else 0.0
    print(f"  Classes evaluated: {len(auc_scores)}/{all_targets.shape[1]}")
    print(f"  Mean ROC-AUC: {mean_auc:.4f}")
    return mean_auc


def run_inference_pipeline(model, label_encoder, config):
    test_dir = config.data_dir / "test_soundscapes"
    if not test_dir.exists():
        print("Test directory not found. Skipping inference.")
        return None
    test_files = list(test_dir.glob("*.ogg"))
    if not test_files:
        print("No test soundscapes found. Skipping inference.")
        return None
    print(f"Found {len(test_files)} test soundscapes")
    test_dataset = SoundscapeDataset(test_files, config)
    test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False,
                             num_workers=config.num_workers, pin_memory=True)
    predictor = Predictor(model, config, label_encoder)
    submission = predictor.predict_soundscapes(test_loader)
    submission_path = config.output_dir / "submission.csv"
    submission.to_csv(submission_path, index=False)
    print(f"Submission saved → {submission_path}")
    return submission


print("All classes and helpers defined successfully.")


All classes and helpers defined successfully.


Section 1 — Train & Save Models¶
Only run this section once (or when you want to retrain). It trains both models, saves their weights to /kaggle/working/, and pickles the label encoder.

When TRAIN_MODE = False this cell is skipped entirely — safe to run on notebook open.

In [3]:
# ─── Set to False to skip training and jump straight to evaluation ───
TRAIN_MODE = False

if TRAIN_MODE:
    print("=" * 60)
    print("BirdCLEF 2026 — Training Pipeline")
    print("=" * 60)

    # 1. Load & clean data
    cleaner = DataCleaner(config)
    train_df, taxonomy_df, soundscape_labels_df = cleaner.load_metadata()
    train_df = cleaner.clean_train_data(train_df)
    cleaner.analyze_class_distribution(train_df)

    # 2. Label encoder
    label_encoder = LabelEncoder()
    label_encoder.fit(taxonomy_df["primary_label"].unique())
    print(f"\nEncoded {len(label_encoder.classes_)} species")

    # 3. Folds
    train_df = create_folds(train_df, n_folds=5)
    fold = 0
    train_fold_df = train_df[train_df["fold"] != fold]
    val_fold_df   = train_df[train_df["fold"] == fold]
    print(f"Fold {fold}: train={len(train_fold_df)}  val={len(val_fold_df)}")

    # 4. Datasets & loaders
    train_dataset = BirdCLEFTrainDataset(
        train_fold_df, config, label_encoder, config.data_dir / "train_audio", augment=True
    )
    class_counts = train_fold_df["primary_label"].value_counts()
    sample_weights = train_fold_df["primary_label"].map(lambda x: 1.0 / class_counts[x]).values
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, sampler=sampler,
                              num_workers=config.num_workers, pin_memory=True)
    val_loader = build_val_loader(val_fold_df, label_encoder, config)

    # 5. Train CNN
    print("\n" + "=" * 60)
    print("Training CNN Model (EfficientNet)")
    print("=" * 60)
    cnn_model = BirdCLEFCNN(config)
    cnn_trainer = Trainer(cnn_model, config, criterion=FocalLoss())
    trained_cnn = cnn_trainer.fit(train_loader, val_loader,
                                   save_path=config.output_dir / "cnn_best.pth")

    # 6. Train Hybrid
    print("\n" + "=" * 60)
    print("Training Hybrid CNN-Transformer Model")
    print("=" * 60)
    hybrid_model = HybridModel(config)
    hybrid_trainer = Trainer(hybrid_model, config, criterion=FocalLoss())
    trained_hybrid = hybrid_trainer.fit(train_loader, val_loader,
                                         save_path=config.output_dir / "hybrid_best.pth")

    # 7. Save label encoder
    with open(config.output_dir / "label_encoder.pkl", "wb") as f:
        pickle.dump(label_encoder, f)
    print("\nLabel encoder saved → label_encoder.pkl")

    # 8. Verify saved files
    print("\nSaved files:")
    for fname in ["cnn_best.pth", "hybrid_best.pth", "label_encoder.pkl"]:
        fpath = config.output_dir / fname
        if fpath.exists():
            print(f"  ✓ {fname}  ({fpath.stat().st_size / 1e6:.1f} MB)")
        else:
            print(f"  ✗ {fname}  NOT FOUND")
else:
    print("TRAIN_MODE=False — skipping training. Jump to Section 2 to load saved models.")


TRAIN_MODE=False — skipping training. Jump to Section 2 to load saved models.


Section 2 — Load Saved Models & Evaluate
Run this every time you reopen the notebook. It loads the weights saved in Section 1 and evaluates both models — no retraining.

Make sure /kaggle/working/cnn_best.pth, hybrid_best.pth, and label_encoder.pkl exist (they persist across Kaggle sessions as notebook outputs).

In [4]:
# ============================================================
# Step 1 — Load label encoder and rebuild data splits
# ============================================================
with open("/kaggle/input/datasets/katie090902/trained-models/label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)
print(f"Label encoder loaded: {len(label_encoder.classes_)} species")

# Rebuild metadata & folds (needed to reconstruct val_fold_df)
cleaner = DataCleaner(config)
train_df, taxonomy_df, soundscape_labels_df = cleaner.load_metadata()
train_df = cleaner.clean_train_data(train_df)
train_df = create_folds(train_df, n_folds=5)

fold = 0
val_fold_df = train_df[train_df["fold"] == fold]
print(f"Validation samples (fold {fold}): {len(val_fold_df)}")

# ============================================================
# Step 2 — Build val_loader
# ============================================================
val_loader = build_val_loader(val_fold_df, label_encoder, config)
print("Validation loader ready.")

# ============================================================
# Step 3 — Instantiate models and load saved weights
# ============================================================
trained_cnn    = BirdCLEFCNN(config)
trained_hybrid = HybridModel(config)

cnn_ckpt    = torch.load("/kaggle/input/datasets/katie090902/trained-models/cnn_best.pth",    map_location=config.device)
hybrid_ckpt = torch.load("/kaggle/input/datasets/katie090902/trained-models/hybrid_best.pth", map_location=config.device)

trained_cnn.load_state_dict(cnn_ckpt["model_state_dict"])
trained_hybrid.load_state_dict(hybrid_ckpt["model_state_dict"])

trained_cnn    = trained_cnn.to(config.device).eval()
trained_hybrid = trained_hybrid.to(config.device).eval()

print(f"CNN loaded    — best epoch: {cnn_ckpt['epoch']+1}, val_loss: {cnn_ckpt['val_loss']:.4f}")
print(f"Hybrid loaded — best epoch: {hybrid_ckpt['epoch']+1}, val_loss: {hybrid_ckpt['val_loss']:.4f}")
print("\nModels ready for evaluation — no retraining needed!")


Label encoder loaded: 234 species
Loading metadata...
  Train CSV: 35549 recordings
  Taxonomy: 234 species
  Soundscape labels: 1478 segments

Cleaning training data...
  Cleaned: 35549 -> 34144 recordings
Validation samples (fold 0): 6829
Validation loader ready.
CNN loaded    — best epoch: 3, val_loss: 0.0018
Hybrid loaded — best epoch: 3, val_loss: 0.0023

Models ready for evaluation — no retraining needed!


In [5]:
# ============================================================
# Step 4 — Evaluate both models on the validation fold
# ============================================================
print("\n" + "=" * 60)
print("Model Comparison — ROC-AUC Scores")
print("=" * 60)

print("\nCNN Model (EfficientNet):")
cnn_auc = evaluate_model(trained_cnn, val_loader, config)

print("\nHybrid CNN-Transformer Model:")
hybrid_auc = evaluate_model(trained_hybrid, val_loader, config)

print("\n" + "=" * 60)
print(f"CNN AUC:    {cnn_auc:.4f}")
print(f"Hybrid AUC: {hybrid_auc:.4f}")
better = "CNN" if cnn_auc >= hybrid_auc else "Hybrid"
print(f"Better model: {better}")
print("=" * 60)



Model Comparison — ROC-AUC Scores

CNN Model (EfficientNet):


Evaluating: 100%|██████████| 107/107 [06:26<00:00,  3.61s/it]


  Classes evaluated: 201/234
  Mean ROC-AUC: 0.8614

Hybrid CNN-Transformer Model:


Evaluating: 100%|██████████| 107/107 [05:00<00:00,  2.81s/it]


  Classes evaluated: 201/234
  Mean ROC-AUC: 0.7306

CNN AUC:    0.8614
Hybrid AUC: 0.7306
Better model: CNN


In [6]:
# # ============================================================
# # Step 5 — Generate Test Predictions for Submission
# # ============================================================
# import os
# import glob

# # Pick the better model (or ensemble both)
# if cnn_auc >= hybrid_auc:
#     best_model = trained_cnn
#     print("Using CNN model for submission")
# else:
#     best_model = trained_hybrid
#     print("Using Hybrid model for submission")

# # Optional: Ensemble both (usually better!)
# # best_model = None  # we'll average predictions below

# best_model.eval()
# best_model.to(config.device)

# # Load sample submission to get the exact row_ids and column order
# sample_sub = pd.read_csv("/kaggle/input/competitions/birdclef-2026/sample_submission.csv")
# species_cols = [c for c in sample_sub.columns if c != "row_id"]

# # Build test datasetes"

# test_soundscape_dir = Path("/kaggle/input/competitions/birdclef-2026/test_soundscapes")
# test_files = sorted(glob.glob(str(test_soundscape_dir / "*.ogg")))
# print(f"Directory exists: {test_soundscape_dir.exists()}")
# print(f"Found {len(test_files)} test soundscape files")
# all_rows = []

# with torch.no_grad():
#     for filepath in tqdm(test_files, desc="Inference"):
#         filename = Path(filepath).stem  # e.g. BC2026_Test_0001_S05_20250227_010002

#         # Load audio
#         audio, sr = librosa.load(filepath, sr=config.sample_rate, mono=True)

#         # Split into 5-second chunks
#         segment_len = int(config.sample_rate * config.segment_duration)
#         num_segments = max(1, len(audio) // segment_len)

#         for seg_idx in range(num_segments):
#             start = seg_idx * segment_len
#             chunk = audio[start: start + segment_len]

#             # Pad if needed
#             if len(chunk) < segment_len:
#                 chunk = np.pad(chunk, (0, segment_len - len(chunk)))

#             # Build row_id: filename + end_time in seconds
#             end_time = int((seg_idx + 1) * config.segment_duration)
#             row_id = f"{filename}_{end_time}"

#             # Compute mel spectrogram
#             mel = librosa.feature.melspectrogram(
#                 y=chunk, sr=config.sample_rate,
#                 n_fft=config.n_fft, hop_length=config.hop_length,
#                 n_mels=config.n_mels, fmin=config.fmin, fmax=config.fmax
#             )
#             mel_db = librosa.power_to_db(mel, ref=np.max).astype(np.float32)
#             mel_tensor = torch.tensor(mel_db).unsqueeze(0).unsqueeze(0).to(config.device)

#             # Run inference
#             logits = best_model(mel_tensor)
#             probs = torch.sigmoid(logits).cpu().numpy()[0]

#             row = {"row_id": row_id}
#             for species, prob in zip(label_encoder.classes_, probs):
#                 row[species] = prob
#             all_rows.append(row)

# # Build submission DataFrame
# if len(all_rows) == 0:
#     print("WARNING: No predictions generated — using uniform probabilities as fallback.")
#     submission_df = sample_sub.copy()
#     for col in species_cols:
#         submission_df[col] = 1 / len(species_cols)
# else:
#     submission_df = pd.DataFrame(all_rows)
#     submission_df = submission_df.set_index("row_id").reindex(
#         columns=species_cols, fill_value=1/len(species_cols)
#     ).reset_index()
#     submission_df = sample_sub[["row_id"]].merge(submission_df, on="row_id", how="left").fillna(1/len(species_cols))

# submission_df.to_csv("/kaggle/working/submission.csv", index=False)
# print(f"Submission saved! Shape: {submission_df.shape}")
# print(submission_df.head(3))

In [7]:
# ============================================================
# Step 5 — Generate Test Predictions for Submission
# ============================================================

# Pick best model
if cnn_auc >= hybrid_auc:
    best_model = trained_cnn
    print("Using CNN model for submission")
else:
    best_model = trained_hybrid
    print("Using Hybrid model for submission")

# Run inference using the built-in pipeline
submission = run_inference_pipeline(best_model, label_encoder, config)

# Fallback if test files are empty (during local dev)
if submission is None:
    print("WARNING: No test files found — saving fallback submission.")
    sample_sub = pd.read_csv("/kaggle/input/competitions/birdclef-2026/sample_submission.csv")
    species_cols = [c for c in sample_sub.columns if c != "row_id"]
    submission = sample_sub.copy()
    for col in species_cols:
        submission[col] = 1 / len(species_cols)
    submission.to_csv("/kaggle/working/submission.csv", index=False)

print(f"Done! Shape: {submission.shape}")
print(submission.head(3))

Using CNN model for submission
No test soundscapes found. Skipping inference.
Done! Shape: (3, 235)
                                    row_id   1161364    116570   1176823  \
0   BC2026_Test_0001_S05_20250227_010002_5  0.004274  0.004274  0.004274   
1  BC2026_Test_0001_S05_20250227_010002_10  0.004274  0.004274  0.004274   
2  BC2026_Test_0001_S05_20250227_010002_15  0.004274  0.004274  0.004274   

    1491113   1595929    209233     22930     22956     22961  ...   whnjay1  \
0  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274  ...  0.004274   
1  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274  ...  0.004274   
2  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274  ...  0.004274   

     whtdov   whwpic1    y00678    yebcar   yebela1    yecmac    yecpar  \
0  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274   
1  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274   
2  0.004274  0.004274  0.004274  0.004274  0.0042